# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sumit-M-Poonia/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

###  Data Contract Definitions

* **Unit of Analysis (Grain):** One row represents one URL on a client domain evaluated over a 90-day window as of a specific decision date (e.g., mid-panel snapshot `2026-03`).
* **Table(s) Used:** `content_refresh_anonymized.csv` (representing the warehouse panel slice).
* **Time Window:** Mid-panel observation period (`month == '2026-03'`) with 90-day lookback metrics prior to the decision point.
* **Target / Proxy Predictor:** Binary proxy label `is_declining` ($1$ if `trend_direction == 'down'`, else $0$).
* **Deliberately Excluded Column:** Future post-decision traffic/impression metrics (e.g., traffic recorded after the decision month) to prevent target leakage.

In [8]:
import os
import pandas as pd

# Fallback direct URL for Google Colab environments
raw_github_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

# Relative paths for local repository setups
possible_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv"
]

# Find available path or use raw GitHub URL
data_source = next((p for p in possible_paths if os.path.exists(p)), raw_github_url)

# Load dataset
df = pd.read_csv(data_source)

# 1. Filter to observation time window if available
if 'month' in df.columns:
    df = df[df['month'] == '2026-03'].copy()

# 2. Verify Grain (Unit of Analysis: 1 row = 1 unique record)
if 'url' in df.columns:
    assert df['url'].is_unique, "Data contract error: 'url' is not unique per row!"
    print("Data Contract Verified: Grain is unique per 'url'.")
else:
    assert df.index.is_unique, "Data contract error: DataFrame index is not unique!"
    print("Data Contract Verified: Grain is unique per index row.")

print(f"Total rows: {len(df):,}")

# 3. Target / Proxy Creation
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# 4. Exclude post-decision metrics to prevent target leakage
leakage_cols = [c for c in df.columns if 'future' in c.lower() or 'post' in c.lower() or 't+90' in c.lower()]
if leakage_cols:
    df.drop(columns=leakage_cols, inplace=True)
    print(f"Removed leakage columns: {leakage_cols}")

print("\nData Contract Summary:")
print(f"- Shape: {df.shape}")
print(f"- Target distribution ('is_declining'):\n{df['is_declining'].value_counts(normalize=True).round(3)}")

Data Contract Verified: Grain is unique per index row.
Total rows: 30,000

Data Contract Summary:
- Shape: (30000, 45)
- Target distribution ('is_declining'):
is_declining
1    0.542
0    0.458
Name: proportion, dtype: float64


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

###  Data Contract Definitions

* **Unit of Analysis (Grain):** One row represents one URL on a client domain evaluated over a 90-day window as of a specific decision date (e.g., mid-panel snapshot `2026-03`).
* **Table(s) Used:** `content_refresh_anonymized.csv` (representing the warehouse panel slice).
* **Time Window:** Mid-panel observation period (`month == '2026-03'`) with 90-day lookback metrics prior to the decision point.
* **Target / Proxy Predictor:** Binary proxy label `is_declining` ($1$ if `trend_direction == 'down'`, else $0$).
* **Deliberately Excluded Column:** Future post-decision traffic/impression metrics (e.g., traffic recorded after the decision month) to prevent target leakage.

In [10]:
import os
import pandas as pd

# Fallback direct URL for Google Colab environments
raw_github_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

# Relative paths for local repository setups
possible_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv"
]

# Find available path or use raw GitHub URL
data_source = next((p for p in possible_paths if os.path.exists(p)), raw_github_url)

# Load dataset
df = pd.read_csv(data_source)

# 1. Filter to observation time window if available
if 'month' in df.columns:
    df = df[df['month'] == '2026-03'].copy()

# 2. Verify Grain (Unit of Analysis: 1 row = 1 unique record)
if 'url' in df.columns:
    assert df['url'].is_unique, "Data contract error: 'url' is not unique per row!"
    print("Data Contract Verified: Grain is unique per 'url'.")
else:
    assert df.index.is_unique, "Data contract error: DataFrame index is not unique!"
    print("Data Contract Verified: Grain is unique per index row.")

print(f"Total rows: {len(df):,}")

# 3. Target / Proxy Creation
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# 4. Exclude post-decision metrics to prevent target leakage
leakage_cols = [c for c in df.columns if 'future' in c.lower() or 'post' in c.lower() or 't+90' in c.lower()]
if leakage_cols:
    df.drop(columns=leakage_cols, inplace=True)
    print(f"Removed leakage columns: {leakage_cols}")

print("\nData Contract Summary:")
print(f"- Shape: {df.shape}")
print(f"- Target distribution ('is_declining'):\n{df['is_declining'].value_counts(normalize=True).round(3)}")

Data Contract Verified: Grain is unique per index row.
Total rows: 30,000

Data Contract Summary:
- Shape: (30000, 45)
- Target distribution ('is_declining'):
is_declining
1    0.542
0    0.458
Name: proportion, dtype: float64


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
import os
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# --- DATA LOADING WITH FALLBACK ---
raw_github_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

possible_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv"
]

data_source = next((p for p in possible_paths if os.path.exists(p)), raw_github_url)
df = pd.read_csv(data_source)

print("=== PART 1: THREE VERIFICATION QUERIES ===")

# Query 1: Verify Grain (1 Row = 1 URL Record)
grain_unique = df.index.is_unique
print(f"Query 1 (Grain Check): Unique Index Constraint Valid = {grain_unique} | Total Rows: {len(df):,}")

# Query 2: Slice Row Count and Age Span
total_rows = len(df)
min_age = df['content_age_days'].min()
max_age = df['content_age_days'].max()
print(f"Query 2 (Slice Summary): {total_rows:,} rows | Content age span: {min_age} to {max_age} days")

# Query 3: Availability Check (Filtering IS TRUE)
avail_mask = (df['impressions_90d'] > 0) & (df['avg_position'].notna())
surviving_rows = avail_mask.sum()
print(f"Query 3 (Availability Check): `(impressions_90d > 0) IS TRUE` -> {surviving_rows:,} / {total_rows:,} rows survived ({surviving_rows/total_rows*100:.1f}%)\n")


print("=== PART 2: FIVE FEATURE FRAME ===")

# Define 5 Honest Features
feature_cols = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr"]
df["target_is_declining"] = (df["trend_direction"] == "down").astype(int)

# Availability justification printout
feature_justifications = {
    "content_age_days": "Knowable at decision moment because published date is an immutable historical timestamp.",
    "days_since_last_update": "Knowable at decision moment because past CMS edit timestamps are fully logged.",
    "impressions_90d": "Knowable at decision moment because Google Search Console logs past 90-day totals prior to today.",
    "avg_position": "Knowable at decision moment because past 90-day search rank averages are already recorded.",
    "ctr": "Knowable at decision moment because historical clicks divided by impressions are calculated prior to decision date."
}

for col in feature_cols:
    print(f"• {col}: {feature_justifications[col]}")


print("\n=== PART 3: THE LEAKAGE TRAP EXPERIMENT ===")

X_honest = df[feature_cols].fillna(0)
y = df["target_is_declining"]

X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.2, random_state=42)

# 1. Honest Baseline Model
clf_honest = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_honest.fit(X_tr, y_tr)
honest_acc = accuracy_score(y_te, clf_honest.predict(X_te))
print(f"1. Honest Baseline Model Accuracy: {honest_acc:.4f}")

# 2. Injecting The Trap (Target-derived leaky feature)
df["leaky_target_column"] = df["target_is_declining"] # Direct label leakage
X_leaky = df[feature_cols + ["leaky_target_column"]].fillna(0)

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
clf_leaky = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_leaky.fit(X_tr_l, y_tr_l)
leaky_acc = accuracy_score(y_te_l, clf_leaky.predict(X_te_l))
print(f"2. LEAKY Model Accuracy (The Trap!):  {leaky_acc:.4f}  <-- Artificial Perfect Score!")

# 3. Removing The Trap
X_cleaned = X_leaky.drop(columns=["leaky_target_column"])
clf_cleaned = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_cleaned.fit(X_tr, y_tr)
restored_acc = accuracy_score(y_te, clf_cleaned.predict(X_te))
print(f"3. Restored Honest Model Accuracy:     {restored_acc:.4f}  <-- Trap Removed")

=== PART 1: THREE VERIFICATION QUERIES ===
Query 1 (Grain Check): Unique Index Constraint Valid = True | Total Rows: 30,000
Query 2 (Slice Summary): 30,000 rows | Content age span: 90 to 564 days
Query 3 (Availability Check): `(impressions_90d > 0) IS TRUE` -> 30,000 / 30,000 rows survived (100.0%)

=== PART 2: FIVE FEATURE FRAME ===
• content_age_days: Knowable at decision moment because published date is an immutable historical timestamp.
• days_since_last_update: Knowable at decision moment because past CMS edit timestamps are fully logged.
• impressions_90d: Knowable at decision moment because Google Search Console logs past 90-day totals prior to today.
• avg_position: Knowable at decision moment because past 90-day search rank averages are already recorded.
• ctr: Knowable at decision moment because historical clicks divided by impressions are calculated prior to decision date.

=== PART 3: THE LEAKAGE TRAP EXPERIMENT ===
1. Honest Baseline Model Accuracy: 0.6482
2. LEAKY Model A

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

###    Named Limitation of the Slice

* **Unobserved External Dynamics (Algorithm & Competitor Shifts):**
  * This data slice relies exclusively on internal 90-day historical performance metrics per URL.
  * It does not account for external shocks such as Google Core Algorithm updates, unobserved competitor content launches, or macro seasonality.
  * **Operational Risk:** A page flagged as "healthy" based on past impressions may suddenly drop tomorrow if a major competitor publishes superior content or search intent shifts.

In [14]:
# Code check for Section 4: Document and log slice limitations & missing external signals
missing_external_signals = [
    'serp_competitor_count',
    'core_algorithm_update_flag',
    'search_volume_seasonality_index',
    'backlink_loss_count'
]

available_cols = set(df.columns)
unobserved_in_slice = [col for col in missing_external_signals if col not in available_cols]

print("--- Limitation Check: External Signals Audit ---")
print(f"Unobserved external variables in current data slice: {len(unobserved_in_slice)} / {len(missing_external_signals)}")
for signal in unobserved_in_slice:
    print(f" ❌ Missing: {signal}")

print("\nImpact Note: Model predictions assume macro stability and evaluate relative decay based solely on internal trajectory.")

--- Limitation Check: External Signals Audit ---
Unobserved external variables in current data slice: 4 / 4
 ❌ Missing: serp_competitor_count
 ❌ Missing: core_algorithm_update_flag
 ❌ Missing: search_volume_seasonality_index
 ❌ Missing: backlink_loss_count

Impact Note: Model predictions assume macro stability and evaluate relative decay based solely on internal trajectory.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-Check

- [x] **5 Contract Answers:** Plain-words definitions provided for grain, tables, time window, proxy label, and deliberate exclusions.
- [x] **3 Verification Queries:** Ran and displayed grain check, row count/date span, and availability filter with `IS TRUE`.
- [x] **5 Feature Frame:** Built 5 features with explicit "knowable at decision moment because..." justification lines.
- [x] **Leakage Trap Experiment:** Executed baseline model, demonstrated artificial perfect score via label leakage, and restored honest model score after dropping the leaky feature.
- [x] **Named Limitation:** Contextualized boundaries regarding external search engine updates and competitor actions.